In [2]:
import numpy as np
import pandas as pd 
import sklearn

In [3]:
df_rabat=pd.read_csv('/home/yazid/housing/scraping/csv/rabat.csv')
df_sale=pd.read_csv('/home/yazid/housing/scraping/csv/sale.csv')
df_temara=pd.read_csv('/home/yazid/housing/scraping/csv/temara.csv')
df_casa=pd.read_csv('/home/yazid/housing/scraping/csv/casa.csv')
df_marrakech=pd.read_csv('/home/yazid/housing/scraping/csv/marrakech.csv')

In [4]:
df_rabat['City']='Rabat'
df_sale['City']='Sale'
df_temara['City']='Temara'
df_casa['City']='Casa'
df_marrakech['City']='Marrakech'

In [5]:
len(df_rabat),len(df_sale),len(df_temara),len(df_casa),len(df_marrakech)

(940, 940, 920, 1880, 1880)

In [6]:
df=pd.concat([df_rabat,df_sale,df_temara,df_casa,df_marrakech],ignore_index=True)

In [7]:
df.replace('Not Specified',np.nan,inplace=True)

,surface,chambres,sdb,etage,location,prix,City
0,171 m²,3 chambres,2 sdbs,Étage 4,"Rabat, Hay Riad",5 500 000,Rabat
1,140 m²,3 chambres,2 sdbs,NaN,"Rabat, Riyad",3 600 000,Rabat
2,122 m²,3 chambres,3 sdbs,Étage 3,"Rabat, Hay Riad",3 700 000,Rabat
3,138 m²,3 chambres,2 sdbs,Étage 2,"Hay Riad, Rabat",NaN,Rabat
4,137 m²,3 chambres,2 sdbs,Étage 2,"Hay Riad, Rabat",NaN,Rabat
...,...,...,...,...,...,...,...
6555,NaN,NaN,NaN,NaN,NaN,470 000,Marrakech
6556,118 m²,2 chambres,1 sdb,Étage 4,"Autre secteur, Casablanca",2 100 000,Marrakech
6557,125 m²,3 chambres,2 sdbs,Étage 3,"Hay Hassani, Casablanca",1 350 000,Marrakech
6558,77 m²,2 chambres,1 sdb,Étage 4,"Sidi Bernoussi, Casablanca",790 000,Marrakech


In [8]:
numerical_cols=['surface','chambres','sdb','etage','prix']
df['etage']=df['etage'].replace({'Rez de chaussée':0})
df['surface']=df['surface'].astype(str).replace(',','.',regex=False).str.extract(r'(\d+\.?\d*)')[0].astype(float)
df['chambres']=df['chambres'].astype(str).str.extract(r'(\d+)')[0].astype(float)
df['sdb']=df['sdb'].astype(str).str.extract(r'(\d+)')[0].astype(float)
df['etage']=df['etage'].astype(str).str.extract(r'(\d+)')[0].astype(float)
df['prix']=df['prix'].astype(str).str.replace(r'[\s\xa0]','',regex=True).str.extract(r'(\d+)')[0].astype(float)

In [9]:
df['surface'].max()

np.float64(1953400.0)

In [10]:
df.dropna(subset=['prix'],inplace=True)

In [11]:
df['location']=df['location'].fillna(df['City'])

In [12]:
(df.isna().sum()/len(df))*100

surface     36.559322
chambres    21.661017
sdb         24.067797
etage       23.491525
location     0.000000
prix         0.000000
City         0.000000
dtype: float64

In [13]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
def impute_by_city(group):
    numeric_cols=['surface','chambres','sdb','etage']
    max_col=group[numeric_cols].max().values
    imputer=IterativeImputer(max_iter=10,random_state=42,min_value=0,max_value=max_col)
    group[numeric_cols]=imputer.fit_transform(group[numeric_cols])
    return group

In [14]:
df=df.groupby('City',group_keys=False)[df.columns].apply(impute_by_city)

In [15]:
((df.isna().sum())/len(df))*100

surface     0.0
chambres    0.0
sdb         0.0
etage       0.0
location    0.0
prix        0.0
City        0.0
dtype: float64

In [16]:
df.head()

,surface,chambres,sdb,etage,location,prix,City
0,171.0,3.0,2.0,4.000000,"Rabat, Hay Riad",5500000.0,Rabat
1,140.0,3.0,2.0,2.199331,"Rabat, Riyad",3600000.0,Rabat
2,122.0,3.0,3.0,3.000000,"Rabat, Hay Riad",3700000.0,Rabat
5,150.0,3.0,2.0,1.000000,"Agdal, Rabat",3400000.0,Rabat
6,120.0,2.0,2.0,0.000000,"Hay Riad, Rabat",2500000.0,Rabat


Let's start with some feature engineering:

In [17]:
df['etage']=df['etage'].round().astype(float)
df['prix par m2']=df['prix']/df['surface']
df['taille moyenne chambre']=df['surface']/(df['chambres'])
df['sdb par chambre']=df['sdb']/df['chambres']
df['chambres totales']=df['chambres']+df['sdb']
df['est rez de chausee']=(df['etage']==0)
df['log_prix']=np.log1p(df['prix'])
df['log_surface']=np.log1p(df['surface'])

In [18]:
(df.isna().sum()/len(df))*100

surface                   0.0
chambres                  0.0
sdb                       0.0
etage                     0.0
location                  0.0
prix                      0.0
City                      0.0
prix par m2               0.0
taille moyenne chambre    0.0
sdb par chambre           0.0
chambres totales          0.0
est rez de chausee        0.0
log_prix                  0.0
log_surface               0.0
dtype: float64

In [19]:
df.head()

,surface,chambres,sdb,etage,location,prix,City,prix par m2,taille moyenne chambre,sdb par chambre,chambres totales,est rez de chausee,log_prix,log_surface
0,171.0,3.0,2.0,4.0,"Rabat, Hay Riad",5500000.0,Rabat,32163.742690,57.000000,0.666667,5.0,False,15.520259,5.147494
1,140.0,3.0,2.0,2.0,"Rabat, Riyad",3600000.0,Rabat,25714.285714,46.666667,0.666667,5.0,False,15.096445,4.948760
2,122.0,3.0,3.0,3.0,"Rabat, Hay Riad",3700000.0,Rabat,30327.868852,40.666667,1.000000,6.0,False,15.123844,4.812184
5,150.0,3.0,2.0,1.0,"Agdal, Rabat",3400000.0,Rabat,22666.666667,50.000000,0.666667,5.0,False,15.039286,5.017280
6,120.0,2.0,2.0,0.0,"Hay Riad, Rabat",2500000.0,Rabat,20833.333333,60.000000,1.000000,4.0,True,14.731802,4.795791


In [20]:
df['surface'].quantile([0.80,0.85,0.90,0.95,0.99,0.995,0.975,0.9999])

0.8000    2.040000e+02
0.8500    3.814958e+03
0.9000    4.162530e+03
0.9500    4.212908e+03
0.9900    4.212908e+03
0.9950    4.212908e+03
0.9750    4.212908e+03
0.9999    1.953400e+06
Name: surface, dtype: float64

In [23]:
df['prix'].quantile([0.9,0.95,0.975,0.99,0.9925,0.995,0.9975,0.999])

0.9000    3.350000e+06
0.9500    4.100000e+06
0.9750    5.500000e+06
0.9900    9.405900e+06
0.9925    2.600000e+07
0.9950    4.600000e+07
0.9975    1.405050e+08
0.9990    1.895450e+08
Name: prix, dtype: float64

Based on the quantiles we keep only relevant rows:

In [24]:
df=df[(df['prix']<9.5e+06) &(df['surface']<4.3e+03)]

In [25]:
df.head()

,surface,chambres,sdb,etage,location,prix,City,prix par m2,taille moyenne chambre,sdb par chambre,chambres totales,est rez de chausee,log_prix,log_surface
0,171.0,3.0,2.0,4.0,"Rabat, Hay Riad",5500000.0,Rabat,32163.742690,57.000000,0.666667,5.0,False,15.520259,5.147494
1,140.0,3.0,2.0,2.0,"Rabat, Riyad",3600000.0,Rabat,25714.285714,46.666667,0.666667,5.0,False,15.096445,4.948760
2,122.0,3.0,3.0,3.0,"Rabat, Hay Riad",3700000.0,Rabat,30327.868852,40.666667,1.000000,6.0,False,15.123844,4.812184
5,150.0,3.0,2.0,1.0,"Agdal, Rabat",3400000.0,Rabat,22666.666667,50.000000,0.666667,5.0,False,15.039286,5.017280
6,120.0,2.0,2.0,0.0,"Hay Riad, Rabat",2500000.0,Rabat,20833.333333,60.000000,1.000000,4.0,True,14.731802,4.795791


In [26]:
len(df)

5837

In [27]:
df.columns.tolist()

['surface',
 'chambres',
 'sdb',
 'etage',
 'location',
 'prix',
 'City',
 'prix par m2',
 'taille moyenne chambre',
 'sdb par chambre',
 'chambres totales',
 'est rez de chausee',
 'log_prix',
 'log_surface']

Let's extract the neighborhood name:

In [32]:
cities_pattern = r"\b(Rabat|Casablanca|Casa|Salé|Sale|Témara|Temara|Marrakech)\b"
df['location']=df['location'].str.replace(cities_pattern,'',regex=True,case=False).str.replace(r'[^\w\s]','',regex=True).str.strip()

In [43]:
df['location']=df['location'].replace(r'^\s*$',np.nan,regex=True).str.replace('Autre secteur','Autre Secteur').fillna('Autre Secteur')

In [ ]:
df['location']=np.where(df['location']=='Autre Secteur',)

In [44]:
df['location'].value_counts()

location
Autre Secteur    1628
Hay Riad          289
Agdal             254
Oulfa             215
Maarif            179
                 ... 
Hay Inara           2
Talaa               1
Ittihad Arabi       1
Lazrak              1
Messrour 2          1
Name: count, Length: 125, dtype: int64

In [36]:
df['location'].tail(20)

6539              Anfa
6540                  
6541             Oulfa
6542    Sidi Bernoussi
6543      Finance City
6544       Sidi Moumen
6545                  
6547            Maarif
6548             Oulfa
6549            2 Mars
6550                  
6551      Sidi Maarouf
6552        La Gironde
6553            Racine
6554       Mers Sultan
6555                  
6556     Autre secteur
6557       Hay Hassani
6558    Sidi Bernoussi
6559                  
Name: location, dtype: str